# CLEIDS-Edge — Notebook 01: Preprocessing and Feature Engineering

Cleans, encodes, scales, splits, and rebalances (SMOTE) the four benchmark datasets acquired in Notebook 00, saving model-ready arrays to `data/processed/<dataset>/`.

**Executed locally, not in Colab** (CPU-only work, no GPU needed here — Colab is reserved for Notebook 03's actual training). This notebook was validated by running its logic as a standalone script against the real local `data/raw/` datasets; the cells below are that validated pipeline, transcribed for the permanent record.

IoT-23 is not included yet — it was still downloading when this notebook was built. It will be added as an additional section once available, following the same pattern established here.

## 1. Setup

In [ ]:
import os
import gc
import json
import datetime
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE, RandomOverSampler

pd.set_option("future.no_silent_downcasting", True)

RAW = "data/raw"
PROCESSED = "data/processed"
RANDOM_STATE = 42
os.makedirs(PROCESSED, exist_ok=True)

with open("data/dataset_manifest.json") as f:
    acquisition_manifest = json.load(f)
print("Datasets from Notebook 00:", list(acquisition_manifest["datasets"].keys()))

## 2. Shared helpers

One pipeline is reused across all four datasets: clean -> impute (median for numeric, constant `"missing"` for categorical, fit on train only) -> one-hot encode categoricals -> scale numerics (fit on train only) -> encode the multiclass label -> SMOTE the training split -> save.

**Multiclass label encoding is fit on the union of train+val+test**, not train alone, because NSL-KDD's official test set deliberately contains 17 attack types never seen in training (a known NSL-KDD property for testing generalization to novel attacks) — fitting only on train would crash on those labels at encode time.

**Binary label is derived from the (post-SMOTE) multiclass label**, not resampled independently, so the two stay consistent with each other rather than diverging under separate SMOTE runs.

In [ ]:
def clean_frame(df, label_col, categorical_cols, drop_cols):
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    # Zeek-derived logs (TON_IoT) use '-' as a not-applicable/unset placeholder
    # (e.g. DNS/HTTP fields on flows that aren't DNS/HTTP) -- treat as missing,
    # not a literal string, so numeric columns cast cleanly.
    df = df.replace("-", np.nan)
    # Zeek boolean fields (dns_AA, ssl_resumed, etc.) are literal 'T'/'F' strings
    df = df.replace({"T": 1, "F": 0})
    df = df.replace([np.inf, -np.inf], np.nan)
    before = len(df)
    df = df.drop_duplicates()
    dropped_dupes = before - len(df)
    missing_before = int(df.isnull().sum().sum())
    return df, dropped_dupes, missing_before


def build_feature_lists(df, label_col, categorical_cols, drop_cols):
    exclude = set([label_col, *categorical_cols, *drop_cols])
    return [c for c in df.columns if c not in exclude]

In [ ]:
def fit_transform_split(train_df, val_df, test_df, label_col, categorical_cols, drop_cols, benign_label):
    numeric_cols = build_feature_lists(train_df, label_col, categorical_cols, drop_cols)
    cat_cols = [c for c in categorical_cols if c in train_df.columns]

    num_imputer = SimpleImputer(strategy="median")
    cat_imputer = SimpleImputer(strategy="constant", fill_value="missing")

    # cast to float32 before impute/scale -- float64 intermediates on a 2-3M row,
    # 78-column dataset (CICIDS2017) doubled peak memory past what this machine's
    # 7.8GB RAM can hold
    train_num_raw = train_df[numeric_cols].to_numpy(dtype=np.float32)
    val_num_raw = val_df[numeric_cols].to_numpy(dtype=np.float32)
    test_num_raw = test_df[numeric_cols].to_numpy(dtype=np.float32)

    num_imputer.fit(train_num_raw)
    train_num = num_imputer.transform(train_num_raw).astype(np.float32)
    val_num = num_imputer.transform(val_num_raw).astype(np.float32)
    test_num = num_imputer.transform(test_num_raw).astype(np.float32)
    del train_num_raw, val_num_raw, test_num_raw

    scaler = StandardScaler()
    train_num = scaler.fit_transform(train_num).astype(np.float32)
    val_num = scaler.transform(val_num).astype(np.float32)
    test_num = scaler.transform(test_num).astype(np.float32)

    feature_names = list(numeric_cols)

    if cat_cols:
        cat_imputer.fit(train_df[cat_cols])
        train_cat = cat_imputer.transform(train_df[cat_cols])
        val_cat = cat_imputer.transform(val_df[cat_cols])
        test_cat = cat_imputer.transform(test_df[cat_cols])

        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
        train_cat_enc = ohe.fit_transform(train_cat)
        val_cat_enc = ohe.transform(val_cat)
        test_cat_enc = ohe.transform(test_cat)

        feature_names += list(ohe.get_feature_names_out(cat_cols))

        X_train = np.hstack([train_num, train_cat_enc])
        X_val = np.hstack([val_num, val_cat_enc])
        X_test = np.hstack([test_num, test_cat_enc])
    else:
        ohe = None
        X_train, X_val, X_test = train_num, val_num, test_num

    all_labels = pd.concat([train_df[label_col], val_df[label_col], test_df[label_col]]).astype(str)
    label_encoder = LabelEncoder()
    label_encoder.fit(all_labels)

    y_train_multi = label_encoder.transform(train_df[label_col].astype(str))
    y_val_multi = label_encoder.transform(val_df[label_col].astype(str))
    y_test_multi = label_encoder.transform(test_df[label_col].astype(str))

    benign_encoded = label_encoder.transform([benign_label])[0]

    return {
        "X_train": X_train, "X_val": X_val, "X_test": X_test,
        "y_train_multi": y_train_multi, "y_val_multi": y_val_multi, "y_test_multi": y_test_multi,
        "feature_names": feature_names,
        "label_encoder": label_encoder,
        "benign_encoded": int(benign_encoded),
        "num_imputer": num_imputer, "scaler": scaler, "cat_imputer": cat_imputer if cat_cols else None,
        "ohe": ohe,
    }

In [ ]:
def smote_resample(X_train, y_train_multi):
    """Full SMOTE-to-majority. Only safe for datasets small enough that this
    machine's 7.8GB RAM can hold every class oversampled up to the majority count."""
    counts = pd.Series(y_train_multi).value_counts()
    min_count = counts.min()
    class_dist_before = counts.to_dict()

    if min_count < 2:
        sampler = RandomOverSampler(random_state=RANDOM_STATE)
        method = "RandomOverSampler (fallback: at least one class has <2 samples)"
    else:
        k = min(5, min_count - 1)
        sampler = SMOTE(random_state=RANDOM_STATE, k_neighbors=k)
        method = f"SMOTE(k_neighbors={k})"

    X_res, y_res = sampler.fit_resample(X_train, y_train_multi)
    class_dist_after = pd.Series(y_res).value_counts().to_dict()
    return X_res, y_res, method, class_dist_before, class_dist_after


def smote_resample_capped(X_train, y_train_multi, cap):
    """Oversample only classes below `cap`, up to `cap` -- NOT full match to majority.
    Used where the majority class is large enough (millions of rows) that
    SMOTE-to-majority would need tens of millions of synthetic rows and is
    infeasible on this machine's RAM. Classes already >= cap are left as-is."""
    counts = pd.Series(y_train_multi).value_counts()
    dist_before = counts.to_dict()

    to_raise = {cls: cnt for cls, cnt in counts.items() if cnt < cap}
    if not to_raise:
        return X_train, y_train_multi, f"no resampling needed (all classes >= cap={cap})", dist_before, dist_before

    singleton_classes = [cls for cls, cnt in to_raise.items() if cnt < 2]
    smote_classes = {cls: cap for cls, cnt in to_raise.items() if cnt >= 2}

    X_cur, y_cur = X_train, y_train_multi
    method_parts = []

    if singleton_classes:
        ros_strategy = {cls: cap for cls in singleton_classes}
        ros = RandomOverSampler(sampling_strategy=ros_strategy, random_state=RANDOM_STATE)
        X_cur, y_cur = ros.fit_resample(X_cur, y_cur)
        method_parts.append(f"RandomOverSampler(singleton classes={singleton_classes} -> {cap})")

    if smote_classes:
        min_count = min(counts[c] for c in smote_classes)
        k = min(5, min_count - 1)
        smote = SMOTE(sampling_strategy=smote_classes, k_neighbors=k, random_state=RANDOM_STATE)
        X_cur, y_cur = smote.fit_resample(X_cur, y_cur)
        method_parts.append(f"SMOTE(k_neighbors={k}, cap={cap}, classes={sorted(smote_classes.keys())})")

    dist_after = pd.Series(y_cur).value_counts().to_dict()
    return X_cur, y_cur, "; ".join(method_parts), dist_before, dist_after

In [ ]:
def save_dataset(name, result, y_bin_train, y_bin_val, y_bin_test, sm_method, dist_before, dist_after, dedupe_report, missing_report):
    out_dir = os.path.join(PROCESSED, name)
    os.makedirs(out_dir, exist_ok=True)

    np.save(os.path.join(out_dir, "train_X.npy"), result["X_train_res"].astype(np.float32))
    np.save(os.path.join(out_dir, "train_y_multi.npy"), result["y_train_multi_res"].astype(np.int32))
    np.save(os.path.join(out_dir, "train_y_bin.npy"), y_bin_train.astype(np.int8))

    np.save(os.path.join(out_dir, "val_X.npy"), result["X_val"].astype(np.float32))
    np.save(os.path.join(out_dir, "val_y_multi.npy"), result["y_val_multi"].astype(np.int32))
    np.save(os.path.join(out_dir, "val_y_bin.npy"), y_bin_val.astype(np.int8))

    np.save(os.path.join(out_dir, "test_X.npy"), result["X_test"].astype(np.float32))
    np.save(os.path.join(out_dir, "test_y_multi.npy"), result["y_test_multi"].astype(np.int32))
    np.save(os.path.join(out_dir, "test_y_bin.npy"), y_bin_test.astype(np.int8))

    with open(os.path.join(out_dir, "feature_names.json"), "w") as f:
        json.dump(result["feature_names"], f, indent=2)

    label_classes = list(result["label_encoder"].classes_)
    with open(os.path.join(out_dir, "label_classes.json"), "w") as f:
        json.dump({"classes": label_classes, "benign_encoded": result["benign_encoded"]}, f, indent=2)

    joblib.dump({
        "num_imputer": result["num_imputer"], "scaler": result["scaler"],
        "cat_imputer": result["cat_imputer"], "ohe": result["ohe"],
        "label_encoder": result["label_encoder"],
    }, os.path.join(out_dir, "preprocessor.joblib"))

    return {
        "n_features": len(result["feature_names"]),
        "n_classes": len(label_classes),
        "dedupe_rows_dropped": dedupe_report,
        "missing_values_before_impute": missing_report,
        "smote_method": sm_method,
        "shapes": {
            "train": list(result["X_train_res"].shape),
            "val": list(result["X_val"].shape),
            "test": list(result["X_test"].shape),
        },
        "class_distribution_train_before_resample": {str(k): int(v) for k, v in dist_before.items()},
        "class_distribution_train_after_resample": {str(k): int(v) for k, v in dist_after.items()},
    }


preprocessing_manifest = {"generated_at": datetime.datetime.utcnow().isoformat() + "Z", "datasets": {}}


def process_dataset(name, train_df, val_df, test_df, label_col, categorical_cols, drop_cols, benign_label,
                     dedupe_report, missing_report, smote_cap=None):
    print(f"\n=== {name} ===")
    print(f"train={len(train_df):,} val={len(val_df):,} test={len(test_df):,}")

    result = fit_transform_split(train_df, val_df, test_df, label_col, categorical_cols, drop_cols, benign_label)

    if smote_cap is not None:
        X_res, y_res, sm_method, dist_before, dist_after = smote_resample_capped(
            result["X_train"], result["y_train_multi"], smote_cap
        )
    else:
        X_res, y_res, sm_method, dist_before, dist_after = smote_resample(result["X_train"], result["y_train_multi"])
    result["X_train_res"] = X_res
    result["y_train_multi_res"] = y_res
    print(f"SMOTE method: {sm_method}")
    print(f"train rows before resample: {len(result['y_train_multi']):,}, after: {len(y_res):,}")

    y_bin_train = (y_res != result["benign_encoded"]).astype(int)
    y_bin_val = (result["y_val_multi"] != result["benign_encoded"]).astype(int)
    y_bin_test = (result["y_test_multi"] != result["benign_encoded"]).astype(int)

    entry = save_dataset(name, result, y_bin_train, y_bin_val, y_bin_test, sm_method, dist_before, dist_after,
                          dedupe_report, missing_report)
    preprocessing_manifest["datasets"][name] = entry
    print(f"[{name}] saved to {os.path.join(PROCESSED, name)} -- {entry['n_features']} features, {entry['n_classes']} classes")

    del result, X_res, y_res
    gc.collect()

## 3. NSL-KDD

Uses the official `KDDTrain+`/`KDDTest+` split (not a random re-split) -- a validation set is carved from the 10% of `KDDTrain+`. The `.txt` files are headerless; column names are the well-known 41-feature + label + difficulty NSL-KDD schema, applied explicitly. `difficulty` is dropped (not a feature).

**Note:** the official test set contains 17 attack types absent from training (e.g. `worm`, `sqlattack`, `httptunnel`) -- this is a deliberate NSL-KDD design choice to test generalization to novel attacks, not a data error. A classifier trained without seeing these will naturally score zero recall on them; that is expected and should be reported as such, not treated as a bug to fix.

In [ ]:
NSLKDD_COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land",
    "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in", "num_compromised",
    "root_shell", "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate", "dst_host_srv_rerror_rate", "label",
    "difficulty",
]

train_raw = pd.read_csv("data/raw/nsl-kdd/KDDTrain+.txt", header=None, names=NSLKDD_COLUMNS)
test_raw = pd.read_csv("data/raw/nsl-kdd/KDDTest+.txt", header=None, names=NSLKDD_COLUMNS)

train_clean, dd1, mb1 = clean_frame(train_raw, "label", ["protocol_type", "service", "flag"], ["difficulty"])
test_clean, dd2, mb2 = clean_frame(test_raw, "label", ["protocol_type", "service", "flag"], ["difficulty"])

train_df, val_df = train_test_split(train_clean, test_size=0.1, stratify=train_clean["label"], random_state=RANDOM_STATE)

process_dataset(
    "nsl-kdd", train_df, val_df, test_clean,
    label_col="label", categorical_cols=["protocol_type", "service", "flag"], drop_cols=["difficulty"],
    benign_label="normal",
    dedupe_report={"train": dd1, "test": dd2},
    missing_report={"train": mb1, "test": mb2},
)
del train_raw, test_raw, train_clean, test_clean, train_df, val_df
gc.collect()

## 4. CICIDS2017

All 8 daily flow-feature CSVs are concatenated, then split 70/15/15 (stratified) since there's no official train/test partition.

**Memory-constrained decision:** BENIGN dominates at 2,273,097 of 2,830,743 rows, with the rarest class (`Heartbleed`) at just 11 samples. Full SMOTE-to-majority would need ~10s of millions of synthetic rows -- infeasible on this machine's 7.8GB RAM (confirmed by an out-of-memory crash on the first attempt). Instead, minority classes are raised to a **50,000-row ceiling** (classes already above it are left unchanged) -- a standard practical compromise for this scale of imbalance, not a full rebalance. This is a documented assumption, not a silent shortcut.

In [ ]:
cicids_files = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
]
dfs = []
for f in cicids_files:
    d = pd.read_csv(os.path.join("data/raw/cicids2017", f), low_memory=False)
    d.columns = [c.strip() for c in d.columns]
    label_col_vals = d["Label"]
    numeric_part = d.drop(columns=["Label"]).astype(np.float32, errors="ignore")
    d = pd.concat([numeric_part, label_col_vals], axis=1)
    dfs.append(d)
cic_raw = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()

cic_clean, dd, mb = clean_frame(cic_raw, "Label", [], [])
del cic_raw
gc.collect()

cic_trainval, cic_test = train_test_split(cic_clean, test_size=0.15, stratify=cic_clean["Label"], random_state=RANDOM_STATE)
cic_train, cic_val = train_test_split(cic_trainval, test_size=0.1765, stratify=cic_trainval["Label"], random_state=RANDOM_STATE)
del cic_clean, cic_trainval
gc.collect()

process_dataset(
    "cicids2017", cic_train, cic_val, cic_test,
    label_col="Label", categorical_cols=[], drop_cols=[],
    benign_label="BENIGN",
    dedupe_report={"all": dd}, missing_report={"all": mb},
    smote_cap=50_000,
)
del cic_train, cic_val, cic_test
gc.collect()

## 5. UNSW-NB15

**Uses only the canonical pre-split `UNSW_NB15_training-set.csv` / `UNSW_NB15_testing-set.csv`** (82,332 / 175,341 rows), not the four raw headerless flow batches (`UNSW-NB15_1..4.csv`, ~2.5M rows) also present in this Kaggle mirror -- those lack an aligned ground-truth label file here, so using them would mean fabricating labels. This matches how UNSW-NB15 is used in most published baselines. Documented assumption, per project rules on noting deviations.

**Note:** in this mirror, `testing-set.csv` (175,341 rows) is actually larger than `training-set.csv` (82,332 rows) -- reversed from what the names might suggest. Files are used exactly as named/split by their source rather than second-guessed.

`attack_cat` is the multiclass label (`NaN` filled to `"Normal"` for benign rows, confirmed by cross-checking against the binary `label` column). The original `label` column is dropped and re-derived consistently from the (post-SMOTE) multiclass label instead, same as every other dataset here.

In [ ]:
unsw_train_raw = pd.read_csv("data/raw/unsw-nb15/UNSW_NB15_training-set.csv")
unsw_test_raw = pd.read_csv("data/raw/unsw-nb15/UNSW_NB15_testing-set.csv")

unsw_train_clean, dd1, mb1 = clean_frame(unsw_train_raw, "attack_cat", ["proto", "service", "state"], ["id", "label"])
unsw_test_clean, dd2, mb2 = clean_frame(unsw_test_raw, "attack_cat", ["proto", "service", "state"], ["id", "label"])
unsw_train_clean["attack_cat"] = unsw_train_clean["attack_cat"].fillna("Normal")
unsw_test_clean["attack_cat"] = unsw_test_clean["attack_cat"].fillna("Normal")

unsw_train_df, unsw_val_df = train_test_split(
    unsw_train_clean, test_size=0.1, stratify=unsw_train_clean["attack_cat"], random_state=RANDOM_STATE
)

process_dataset(
    "unsw-nb15", unsw_train_df, unsw_val_df, unsw_test_clean,
    label_col="attack_cat", categorical_cols=["proto", "service", "state"], drop_cols=["id", "label"],
    benign_label="Normal",
    dedupe_report={"train": dd1, "test": dd2}, missing_report={"train": mb1, "test": mb2},
)
del unsw_train_raw, unsw_test_raw, unsw_train_clean, unsw_test_clean, unsw_train_df, unsw_val_df
gc.collect()

## 6. TON_IoT

No official split -- stratified 70/15/15 on `type` (multiclass; `label` binary column dropped and re-derived, consistent with the other datasets).

**Zeek log quirks handled in `clean_frame`:** this dataset is derived from Zeek/Bro connection logs, which use `'-'` as a not-applicable placeholder (e.g. DNS/HTTP fields on non-DNS/HTTP flows) and literal `'T'`/`'F'` strings for booleans (`dns_AA`, `ssl_resumed`, etc.) -- both are normalized before numeric casting. `ssl_version`, `ssl_cipher`, `http_method`, `http_version`, and the MIME-type fields are treated as categorical (one-hot encoded) rather than numeric. High-cardinality identifier-ish fields (`ts`, IPs, ports, `dns_query`, `ssl_subject`/`issuer`, `http_uri`, `http_user_agent`, `weird_name`/`weird_addl`) are dropped as non-generalizable.

In [ ]:
ton_raw = pd.read_csv("data/raw/ton-iot/Train_Test_Network.csv")
ton_categorical = ["proto", "service", "conn_state", "ssl_version", "ssl_cipher",
                    "http_method", "http_version", "http_orig_mime_types", "http_resp_mime_types"]
ton_drop = ["ts", "src_ip", "src_port", "dst_ip", "dst_port", "dns_query", "ssl_subject", "ssl_issuer",
            "http_uri", "http_user_agent", "weird_name", "weird_addl", "label"]
ton_clean, dd, mb = clean_frame(ton_raw, "type", ton_categorical, ton_drop)

ton_trainval, ton_test = train_test_split(ton_clean, test_size=0.15, stratify=ton_clean["type"], random_state=RANDOM_STATE)
ton_train, ton_val = train_test_split(ton_trainval, test_size=0.1765, stratify=ton_trainval["type"], random_state=RANDOM_STATE)

process_dataset(
    "ton-iot", ton_train, ton_val, ton_test,
    label_col="type", categorical_cols=ton_categorical, drop_cols=ton_drop,
    benign_label="normal",
    dedupe_report={"all": dd}, missing_report={"all": mb},
)
del ton_raw, ton_clean, ton_trainval, ton_test, ton_train, ton_val
gc.collect()

## 7. Write preprocessing manifest

In [ ]:
with open(os.path.join(PROCESSED, "preprocessing_manifest.json"), "w") as f:
    json.dump(preprocessing_manifest, f, indent=2)

print("Wrote", os.path.join(PROCESSED, "preprocessing_manifest.json"))

## 8. Final summary

Paste this cell's output back for review before Notebook 02 starts.

In [ ]:
print("=" * 70)
print("CLEIDS-Edge -- Notebook 01 Summary")
print("=" * 70)

for name, info in preprocessing_manifest["datasets"].items():
    print(f"\n[{name}]")
    print(f"  features={info['n_features']} classes={info['n_classes']}")
    print(f"  shapes: train={info['shapes']['train']} val={info['shapes']['val']} test={info['shapes']['test']}")
    print(f"  dedupe rows dropped: {info['dedupe_rows_dropped']}")
    print(f"  SMOTE: {info['smote_method']}")

print("\nIoT-23: not yet processed (still downloading when this notebook was built).")
print("Processed arrays saved under data/processed/<dataset>/ (gitignored -- large files, not committed).")
print("Next: Notebook 02 (CLEIDS-Edge architecture definition).")